In [ ]:
# ==========================================
# 📕 STEP 1: INSTALL & MOUNT
# ==========================================
!pip install -q -U torch transformers peft datasets bitsandbytes trl


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer

# Project Paths
project_path = "/content/drive/MyDrive/BlueTeamLab"
data_path = f"{project_path}/data"
model_save_path = f"{project_path}/models/tinyllama_guard_v1"

print(f"✅ GPU Status: {torch.cuda.get_device_name(0)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ GPU Status: Tesla T4


In [ ]:
# ==========================================
# 📕 STEP 2: LOAD ASSETS
# ==========================================

# 1. Load your pre-prepared CSVs as Hugging Face Datasets
print("📥 Loading Data from Drive...")
dataset = load_dataset("csv", data_files={
    "train": f"{data_path}/train.csv",
    "validation": f"{data_path}/val.csv"
})

# 2. Load Model (4-bit QLoRA)
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"🤖 Loading Model: {model_id}...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 3. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Fix padding issue
tokenizer.padding_side = "right" # Fix fp16 overflow issue

print("✅ Model & Data Ready.")

📥 Loading Data from Drive...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

🤖 Loading Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

✅ Model & Data Ready.


In [ ]:
# ==========================================
# 🛠️ TRL LIBRARY DIAGNOSTIC
# ==========================================
import trl
import inspect
from trl import SFTConfig, SFTTrainer

print(f"🔍 INSTALLED TRL VERSION: {trl.__version__}")

# 1. Check SFTConfig Valid Arguments
print("\n📋 VALID ARGUMENTS FOR SFTConfig:")
config_args = list(inspect.signature(SFTConfig.__init__).parameters.keys())
# Check for the suspicious ones
for arg in ["max_seq_length", "max_length", "dataset_text_field", "packing"]:
    if arg in config_args:
        print(f"   ✅ {arg} is supported!")
    else:
        print(f"   ❌ {arg} is NOT supported.")

# 2. Check SFTTrainer Valid Arguments
print("\n📋 VALID ARGUMENTS FOR SFTTrainer:")
trainer_args = list(inspect.signature(SFTTrainer.__init__).parameters.keys())
for arg in ["max_seq_length", "max_length", "dataset_text_field", "packing"]:
    if arg in trainer_args:
        print(f"   ✅ {arg} is supported!")
    else:
        print(f"   ❌ {arg} is NOT supported.")

🔍 INSTALLED TRL VERSION: 0.25.1

📋 VALID ARGUMENTS FOR SFTConfig:
   ❌ max_seq_length is NOT supported.
   ✅ max_length is supported!
   ✅ dataset_text_field is supported!
   ✅ packing is supported!

📋 VALID ARGUMENTS FOR SFTTrainer:
   ❌ max_seq_length is NOT supported.
   ❌ max_length is NOT supported.
   ❌ dataset_text_field is NOT supported.
   ❌ packing is NOT supported.


In [ ]:
# ==========================================
# 📕 STEP 3 & 4: THE ROBUST FIX (TRL v0.13+)
# ==========================================
from trl import SFTConfig, SFTTrainer
import os
os.environ["WANDB_DISABLED"] = "true"

# 1. DEFINE CONFIG with tokenizer processing arguments
sft_config = SFTConfig(
    output_dir="./results",
    dataset_text_field="training_text",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=2e-4,
    fp16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    packing=False,
    max_length=1024,  # New versions use 'max_length' not 'max_seq_length'
)

print("✅ SFTConfig initialized successfully.")

# 2. INITIALIZE TRAINER - tokenizer is now in processing_class
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,  # ✅ Changed from 'tokenizer' to 'processing_class'
    args=sft_config,
)

print("🚀 Starting Training Loop...")
trainer.train()

print("🎉 Training Complete!")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


✅ SFTConfig initialized successfully.
🚀 Starting Training Loop...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.431800,0.462174,0.464301,254052.000000,0.902028
200,0.377200,0.448129,0.412213,505611.000000,0.905187


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: 

🎉 Training Complete!


In [ ]:
# ==========================================
# 📕 STEP 5: SAVE ARTIFACTS
# ==========================================

print(f"💾 Saving adapter to: {model_save_path}")
trainer.model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("✅ Saved. You can now reload this Guardrail anytime!")

💾 Saving adapter to: /content/drive/MyDrive/BlueTeamLab/models/tinyllama_guard_v1
✅ Saved. You can now reload this Guardrail anytime!
